In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# [PART 1] CMT 모델 아키텍처 직접 구현 (CVPR 2022)
# ==========================================
class LPU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
    def forward(self, x):
        return x + self.dwconv(x)

class LMHSA(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.sr_ratio = sr_ratio
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio, groups=dim)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr = None
            self.norm = None

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        if self.sr is not None:
            x_2d = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_sr = self.sr(x_2d).reshape(B, C, -1).permute(0, 2, 1)
            x_sr = self.norm(x_sr)
            kv = self.kv(x_sr).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class IRFFN(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, hidden_features, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(hidden_features)
        self.act1 = nn.GELU()
        self.dwconv = nn.Conv2d(hidden_features, hidden_features, kernel_size=3, padding=1, groups=hidden_features, bias=False)
        self.bn2 = nn.BatchNorm2d(hidden_features)
        self.act2 = nn.GELU()
        self.conv2 = nn.Conv2d(hidden_features, in_features, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_features)

    def forward(self, x):
        residual = x
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.dwconv(x)))
        x = self.bn3(self.conv2(x))
        return x + residual

class CMTBlock(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio, expansion_ratio=4):
        super().__init__()
        self.lpu = LPU(dim)
        self.ln1 = nn.LayerNorm(dim)
        self.lmhsa = LMHSA(dim, num_heads, sr_ratio)
        self.ln2 = nn.LayerNorm(dim)
        self.irffn = IRFFN(dim, int(dim * expansion_ratio))

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.lpu(x)
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm1 = self.ln1(x_flat)
        attn_out = self.lmhsa(x_norm1, H, W)
        x = x + attn_out.transpose(1, 2).reshape(B, C, H, W)
        x_norm2 = self.ln2(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W)
        x = self.irffn(x_norm2)
        return x

class PatchEmbed(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)
        return x_norm.transpose(1, 2).reshape(B, C, H, W)

class CMT_S(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU()
        )
        self.patch_embed1 = PatchEmbed(32, 64)
        self.stage1 = nn.Sequential(*[CMTBlock(64, num_heads=1, sr_ratio=8) for _ in range(3)])
        self.patch_embed2 = PatchEmbed(64, 128)
        self.stage2 = nn.Sequential(*[CMTBlock(128, num_heads=2, sr_ratio=4) for _ in range(3)])
        self.patch_embed3 = PatchEmbed(128, 256)
        self.stage3 = nn.Sequential(*[CMTBlock(256, num_heads=4, sr_ratio=2) for _ in range(16)])
        self.patch_embed4 = PatchEmbed(256, 512)
        self.stage4 = nn.Sequential(*[CMTBlock(512, num_heads=8, sr_ratio=1) for _ in range(3)])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Conv2d(512, 1280, kernel_size=1)
        self.head = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(self.patch_embed1(x))
        x = self.stage2(self.patch_embed2(x))
        x = self.stage3(self.patch_embed3(x))
        x = self.stage4(self.patch_embed4(x))
        x = self.avgpool(x)
        x = self.proj(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x

# ==========================================
# [PART 2] 훈련 함수 정의 (AMP 혼합정밀 적용)
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs, save_path):
    best_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # ✅ AMP: GradScaler 생성 (CUDA일 때만 실제로 활성화)
    use_amp = (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    if use_amp:
        print("\n🔥 본격적인 CMT 훈련을 시작합니다! (AMP 혼합정밀 ON)")
    else:
        print("\n🔥 본격적인 CMT 훈련을 시작합니다! (CPU 모드)")

    for epoch in range(epochs):
        start_time = time.time()

        # 훈련
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for inputs, labels in train_loop:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            # ✅ AMP: autocast 영역 안에서 forward + loss 계산
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            # ✅ AMP: scaler를 통한 backward / step / update
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            train_loop.set_postfix(loss=f"{loss.item():.4f}")  # 배치마다 현재 loss 표시

        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total

        # 검증
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
        with torch.no_grad():
            for inputs, labels in val_loop:
                inputs, labels = inputs.to(device), labels.to(device)
                # ✅ AMP: 검증에서도 autocast 사용 (속도 향상)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        elapsed_time = time.time() - start_time
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Time: {elapsed_time:.0f}s | "
              f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  🌟 최고 성능 갱신! 모델 저장됨: {save_path}")

    print(f"\n🎉 훈련 종료! 최고 검증 정확도: {best_acc:.4f}")
    return history

def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('CMT Model Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('CMT Model Loss')
    plt.legend()

    graph_path = os.path.join(save_dir, "training_log.png")
    plt.savefig(graph_path)
    print(f"📊 학습 그래프가 저장되었습니다: {graph_path}")
    plt.show()

# 채널 수가 다른 이미지(흑백/RGBA)가 섞여 있어도 안전하게 3채널 RGB로 강제 변환
def rgb_loader(path):
    return Image.open(path).convert("RGB")

# ==========================================
# [PART 3] 메인 실행부 (Windows 멀티프로세싱 에러 완벽 차단)
# ==========================================
if __name__ == "__main__":
    # 🚨 이 아래에 있는 코드들은 오직 '메인 관리자'만 실행합니다! (프리징 완벽 해결)
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"

    if os.path.exists(PATH_LOCAL):
        DATA_DIR = PATH_LOCAL
    elif os.path.exists(PATH_ONEDRIVE):
        DATA_DIR = PATH_ONEDRIVE
    else:
        print("❌ 데이터 파이프라인 폴더를 찾을 수 없습니다. 경로를 확인해주세요.")
        exit()

    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_cmt_model.pt")

    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 1e-4

    # 여전히 메모리 문제로 멈춘다면 NUM_WORKERS를 0으로 두세요.
    NUM_WORKERS = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 학습 장치: {device} 로 구동됩니다.")

    # ✅ 핵심 수정: Resize 추가 (이미지 크기를 224x224로 통일해야 배치 생성이 가능)
    common_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("📂 데이터셋을 불러오는 중입니다...")
    train_dataset = datasets.ImageFolder(
        root=os.path.join(DATA_DIR, 'train'),
        transform=common_transform,
        loader=rgb_loader
    )
    val_dataset = datasets.ImageFolder(
        root=os.path.join(DATA_DIR, 'val'),
        transform=common_transform,
        loader=rgb_loader
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    NUM_CLASSES = len(train_dataset.classes)
    print(f"✅ 총 {NUM_CLASSES}개의 의류 클래스를 감지했습니다.")

    print("🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...")
    model = CMT_S(num_classes=NUM_CLASSES).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    # 훈련 함수 호출
    history = train_model(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, MODEL_SAVE_PATH)

    # 결과 그래프 그리기
    plot_history(history, DATA_DIR)

🚀 학습 장치: cuda 로 구동됩니다.
📂 데이터셋을 불러오는 중입니다...


C:\Users\user\AppData\Local\Temp\ipykernel_28668\3536816860.py:149: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


✅ 총 22개의 의류 클래스를 감지했습니다.
🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...

🔥 본격적인 CMT 훈련을 시작합니다! (AMP 혼합정밀 ON)


Epoch 1/30 [Train]:   0%|          | 0/3438 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_28668\3536816860.py:168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# 🚨 [성능 폭발 레시피 1] RTX 5070의 텐서 코어를 100% 활용하기 위한 AMP 라이브러리
from torch.cuda.amp import autocast, GradScaler 

# ==========================================
# [PART 1] CMT 모델 아키텍처 직접 구현 (이전과 동일)
# ==========================================
class LPU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
    def forward(self, x): return x + self.dwconv(x)

class LMHSA(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.sr_ratio = sr_ratio
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio, groups=dim)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr, self.norm = None, None

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        if self.sr is not None:
            x_2d = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_sr = self.norm(self.sr(x_2d).reshape(B, C, -1).permute(0, 2, 1))
            kv = self.kv(x_sr).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        x = (attn.softmax(dim=-1) @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class IRFFN(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, hidden_features, 1, bias=False)
        self.bn1, self.act1 = nn.BatchNorm2d(hidden_features), nn.GELU()
        self.dwconv = nn.Conv2d(hidden_features, hidden_features, 3, 1, 1, groups=hidden_features, bias=False)
        self.bn2, self.act2 = nn.BatchNorm2d(hidden_features), nn.GELU()
        self.conv2 = nn.Conv2d(hidden_features, in_features, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_features)

    def forward(self, x):
        return x + self.bn3(self.conv2(self.act2(self.bn2(self.dwconv(self.act1(self.bn1(self.conv1(x))))))))

class CMTBlock(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio, expansion_ratio=4):
        super().__init__()
        self.lpu = LPU(dim)
        self.ln1 = nn.LayerNorm(dim)
        self.lmhsa = LMHSA(dim, num_heads, sr_ratio)
        self.ln2 = nn.LayerNorm(dim)
        self.irffn = IRFFN(dim, int(dim * expansion_ratio))

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.lpu(x)
        attn_out = self.lmhsa(self.ln1(x.flatten(2).transpose(1, 2)), H, W)
        x = x + attn_out.transpose(1, 2).reshape(B, C, H, W)
        return self.irffn(self.ln2(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W))

class PatchEmbed(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, 2, 2)
        self.norm = nn.LayerNorm(out_channels)
    def forward(self, x):
        x = self.proj(x)
        return self.norm(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(x.shape[0], x.shape[1], x.shape[2], x.shape[3])

class CMT_S(nn.Module):
    def __init__(self, num_classes=22):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU()
        )
        self.patch_embed1, self.stage1 = PatchEmbed(32, 64), nn.Sequential(*[CMTBlock(64, 1, 8) for _ in range(3)])
        self.patch_embed2, self.stage2 = PatchEmbed(64, 128), nn.Sequential(*[CMTBlock(128, 2, 4) for _ in range(3)])
        self.patch_embed3, self.stage3 = PatchEmbed(128, 256), nn.Sequential(*[CMTBlock(256, 4, 2) for _ in range(16)])
        self.patch_embed4, self.stage4 = PatchEmbed(256, 512), nn.Sequential(*[CMTBlock(512, 8, 1) for _ in range(3)])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Conv2d(512, 1280, 1)
        self.head = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(self.patch_embed1(x))
        x = self.stage2(self.patch_embed2(x))
        x = self.stage3(self.patch_embed3(x))
        x = self.stage4(self.patch_embed4(x))
        x = self.head(self.proj(self.avgpool(x)).view(x.size(0), -1))
        return x

# ==========================================
# [PART 2] 훈련 함수 정의 (튜닝 부스터 장착)
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, epochs, save_path):
    best_acc = 0.0
    patience_counter = 0
    PATIENCE_LIMIT = 15 # 15번 연속으로 최고 기록을 갱신 못하면 조기 종료
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    print(f"\n🔥 [성능 부스트 모드 ON] 85% 돌파를 향한 훈련을 시작합니다! (최대 {epochs} Epoch)")
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # --- 훈련 (Train) ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Train]")
        
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # 💡 AMP (자동 혼합 정밀도): 텐서 코어를 사용해 연산 속도 2배 향상
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{train_correct/train_total:.4f}"})
            
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        
        # --- 검증 (Validation) ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Val]  ")
        
        with torch.no_grad():
            for inputs, labels in val_pbar:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # 검증 시에도 AMP 사용 가능 (속도 향상)
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                val_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{val_correct/val_total:.4f}"})
                
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        # 💡 스케줄러 업데이트 (보폭 조절)
        scheduler.step()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        elapsed_time = time.time() - start_time
        
        # 현재 학습률 가져오기
        current_lr = optimizer.param_groups[0]['lr']
        print(f"✅ Epoch {epoch+1:03d} 완료 | Time: {elapsed_time:.0f}s | LR: {current_lr:.6f} | "
              f"Train Acc: {epoch_train_acc:.4f} | Val Acc: {epoch_val_acc:.4f}")
        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            patience_counter = 0 # 카운터 리셋
            torch.save(model.state_dict(), save_path)
            print(f"  🌟 최고 성능 갱신! 모델 저장됨: {save_path}\n")
        else:
            patience_counter += 1
            print(f"  ⚠️ 성능 갱신 실패 (연속 {patience_counter}회) - {PATIENCE_LIMIT}회 누적 시 조기 종료\n")
            if patience_counter >= PATIENCE_LIMIT:
                print(f"🛑 모델 성능이 더 이상 오르지 않아 조기 종료(Early Stopping) 합니다!")
                break

    print(f"\n🎉 훈련 최종 종료! 최고 검증 정확도: {best_acc:.4f}")
    return history

def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('CMT Model Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('CMT Model Loss')
    plt.legend()
    
    graph_path = os.path.join(save_dir, "training_log.png")
    plt.savefig(graph_path)
    print(f"📊 학습 그래프가 저장되었습니다: {graph_path}")
    plt.show()

# ==========================================
# [PART 3] 메인 실행부
# ==========================================
if __name__ == "__main__":
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
    DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE

    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_cmt_model.pt")

    # 💡 튜닝 값 셋업
    BATCH_SIZE = 64 # AMP 적용으로 메모리 여유가 생겼으므로 배치 사이즈 2배 업! (속도 대폭 상승)
    EPOCHS = 100    # 성능 한계치를 뽑기 위해 100 에포크로 설정 (조기 종료 기능 탑재)
    LEARNING_RATE = 5e-4 # 스케줄러를 믿고 초기 학습률을 살짝 올려 크게 뜁니다.
    NUM_WORKERS = 0 

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 학습 장치: {device} 로 구동됩니다.")

    common_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("📂 데이터셋을 불러오는 중입니다...")
    train_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'train'), transform=common_transform)
    val_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'val'), transform=common_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    NUM_CLASSES = len(train_dataset.classes)
    
    print("🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...")
    model = CMT_S(num_classes=NUM_CLASSES).to(device)

    # 💡 [성능 폭발 레시피 2] 라벨 스무딩 (label_smoothing=0.1) 적용
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4) # weight_decay 강화
    
    # 💡 [성능 폭발 레시피 3] 코사인 어닐링 스케줄러 (학습률을 예술적으로 조절)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # AMP(자동 혼합 정밀도) 스케일러 생성
    scaler = GradScaler()

    # 훈련 함수 호출
    history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, EPOCHS, MODEL_SAVE_PATH)
    
    # 결과 그래프 그리기
    plot_history(history, DATA_DIR)

In [ ]:
import os
import time
import copy # 🚨 EMA 구현을 위한 모듈 추가
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# 🚨 [성능 폭발 레시피 1] RTX 5070의 텐서 코어를 100% 활용하기 위한 AMP 라이브러리
from torch.cuda.amp import autocast, GradScaler 
# SWA 임포트 삭제 (더 강력한 커스텀 EMA로 대체합니다)

# ==========================================
# [PART 1] CMT 모델 아키텍처 직접 구현 (이전과 동일)
# ==========================================
class LPU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
    def forward(self, x): return x + self.dwconv(x)

class LMHSA(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.sr_ratio = sr_ratio
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio, groups=dim)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr, self.norm = None, None

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        if self.sr is not None:
            x_2d = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_sr = self.norm(self.sr(x_2d).reshape(B, C, -1).permute(0, 2, 1))
            kv = self.kv(x_sr).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        x = (attn.softmax(dim=-1) @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class IRFFN(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, hidden_features, 1, bias=False)
        self.bn1, self.act1 = nn.BatchNorm2d(hidden_features), nn.GELU()
        self.dwconv = nn.Conv2d(hidden_features, hidden_features, 3, 1, 1, groups=hidden_features, bias=False)
        self.bn2, self.act2 = nn.BatchNorm2d(hidden_features), nn.GELU()
        self.conv2 = nn.Conv2d(hidden_features, in_features, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_features)

    def forward(self, x):
        return x + self.bn3(self.conv2(self.act2(self.bn2(self.dwconv(self.act1(self.bn1(self.conv1(x))))))))

class CMTBlock(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio, expansion_ratio=4):
        super().__init__()
        self.lpu = LPU(dim)
        self.ln1 = nn.LayerNorm(dim)
        self.lmhsa = LMHSA(dim, num_heads, sr_ratio)
        self.ln2 = nn.LayerNorm(dim)
        self.irffn = IRFFN(dim, int(dim * expansion_ratio))

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.lpu(x)
        attn_out = self.lmhsa(self.ln1(x.flatten(2).transpose(1, 2)), H, W)
        x = x + attn_out.transpose(1, 2).reshape(B, C, H, W)
        return self.irffn(self.ln2(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W))

class PatchEmbed(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, 2, 2)
        self.norm = nn.LayerNorm(out_channels)
    def forward(self, x):
        x = self.proj(x)
        return self.norm(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(x.shape[0], x.shape[1], x.shape[2], x.shape[3])

class CMT_S(nn.Module):
    def __init__(self, num_classes=22):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU()
        )
        self.patch_embed1, self.stage1 = PatchEmbed(32, 64), nn.Sequential(*[CMTBlock(64, 1, 8) for _ in range(3)])
        self.patch_embed2, self.stage2 = PatchEmbed(64, 128), nn.Sequential(*[CMTBlock(128, 2, 4) for _ in range(3)])
        self.patch_embed3, self.stage3 = PatchEmbed(128, 256), nn.Sequential(*[CMTBlock(256, 4, 2) for _ in range(16)])
        self.patch_embed4, self.stage4 = PatchEmbed(256, 512), nn.Sequential(*[CMTBlock(512, 8, 1) for _ in range(3)])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Conv2d(512, 1280, 1)
        self.head = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(self.patch_embed1(x))
        x = self.stage2(self.patch_embed2(x))
        x = self.stage3(self.patch_embed3(x))
        x = self.stage4(self.patch_embed4(x))
        x = self.head(self.proj(self.avgpool(x)).view(x.size(0), -1))
        return x

# ==========================================
# 💡 [핵심 추가] EMA (지수 이동 평균) 클래스 구현
# ==========================================
class ModelEMA:
    """ 모델 가중치의 지수 이동 평균을 유지하는 클래스 """
    def __init__(self, model, decay=0.999):
        # 원본 모델을 복사하여 EMA 전용 두뇌를 만듭니다.
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        # EMA 가중치는 직접 학습(역전파)하지 않고 비율로만 섞입니다.
        for param in self.ema.parameters():
            param.requires_grad_(False)

    def update(self, model):
        # 매 배치마다 원본 모델의 현재 상태를 아주 조금씩(1-decay) 섞어줍니다.
        with torch.no_grad():
            for ema_param, param in zip(self.ema.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1 - self.decay)

# ==========================================
# [PART 2] 훈련 함수 정의 (튜닝 부스터 장착)
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, epochs, save_path):
    best_acc = 0.0
    patience_counter = 0
    PATIENCE_LIMIT = 15 # 15번 연속으로 최고 기록을 갱신 못하면 조기 종료
    
    # 💡 EMA 객체 생성 (기존 99.9% 유지, 새 지식 0.1% 반영)
    ema = ModelEMA(model, decay=0.999)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    print(f"\n🔥 [성능 부스트 모드 ON] EMA 적용! 85% 돌파를 향한 훈련을 시작합니다! (최대 {epochs} Epoch)")
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # --- 훈련 (Train) ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Train]")
        
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # AMP (자동 혼합 정밀도)
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # 💡 [핵심] 매 배치(Batch)가 끝날 때마다 EMA 두뇌를 업데이트!
            ema.update(model)
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{train_correct/train_total:.4f}"})
            
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        
        # --- 검증 (Validation) ---
        # 💡 검증 시에는 들쭉날쭉한 원본 모델 대신, 안정적인 EMA 모델을 사용합니다!
        ema.ema.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Val]  ")
        
        with torch.no_grad():
            for inputs, labels in val_pbar:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # 💡 TTA (Test-Time Augmentation): 원본과 좌우 반전 이미지를 모두 보고 투표
                inputs_flipped = torch.flip(inputs, dims=[3])
                
                with autocast():
                    outputs_orig = ema.ema(inputs)
                    outputs_flip = ema.ema(inputs_flipped)
                    
                    # 두 예측값의 평균 도출
                    outputs = (outputs_orig + outputs_flip) / 2.0 
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                val_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{val_correct/val_total:.4f}"})
                
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        # 스케줄러 업데이트
        scheduler.step()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        elapsed_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        print(f"✅ Epoch {epoch+1:03d} 완료 | Time: {elapsed_time:.0f}s | LR: {current_lr:.6f} | "
              f"Train Acc: {epoch_train_acc:.4f} | EMA Val Acc: {epoch_val_acc:.4f}")
        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            patience_counter = 0 
            # 💡 최종 저장되는 파일은 불안정한 원본이 아닌 '황금비율 EMA 모델'입니다.
            torch.save(ema.ema.state_dict(), save_path)
            print(f"  🌟 최고 성능 갱신! EMA 모델 저장됨: {save_path}\n")
        else:
            patience_counter += 1
            print(f"  ⚠️ 성능 갱신 실패 (연속 {patience_counter}회) - {PATIENCE_LIMIT}회 누적 시 조기 종료\n")
            if patience_counter >= PATIENCE_LIMIT:
                print(f"🛑 모델 성능이 더 이상 오르지 않아 조기 종료(Early Stopping) 합니다!")
                break

    print(f"\n🎉 훈련 최종 종료! 최고 검증 정확도 (EMA): {best_acc:.4f}")
    return history

def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('CMT Model Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('CMT Model Loss')
    plt.legend()
    
    graph_path = os.path.join(save_dir, "training_log.png")
    plt.savefig(graph_path)
    print(f"📊 학습 그래프가 저장되었습니다: {graph_path}")
    plt.show()

# ==========================================
# [PART 3] 메인 실행부
# ==========================================
if __name__ == "__main__":
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
    DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE

    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_cmt_model.pt")

    # 💡 튜닝 값 셋업
    BATCH_SIZE = 64 # AMP 적용으로 메모리 여유가 생겼으므로 배치 사이즈 2배 업! (속도 대폭 상승)
    EPOCHS = 100    # 성능 한계치를 뽑기 위해 100 에포크로 설정 (조기 종료 기능 탑재)
    LEARNING_RATE = 5e-4 # 스케줄러를 믿고 초기 학습률을 살짝 올려 크게 뜁니다.
    NUM_WORKERS = 0 

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 학습 장치: {device} 로 구동됩니다.")

    common_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("📂 데이터셋을 불러오는 중입니다...")
    train_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'train'), transform=common_transform)
    val_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'val'), transform=common_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    NUM_CLASSES = len(train_dataset.classes)
    
    print("🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...")
    model = CMT_S(num_classes=NUM_CLASSES).to(device)

    # 💡 [성능 폭발 레시피 2] 라벨 스무딩 (label_smoothing=0.1) 적용
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4) # weight_decay 강화
    
    # 💡 [성능 폭발 레시피 3] 코사인 어닐링 스케줄러 (학습률을 예술적으로 조절)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # AMP(자동 혼합 정밀도) 스케일러 생성
    scaler = GradScaler()

    # 훈련 함수 호출
    history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, EPOCHS, MODEL_SAVE_PATH)
    
    # 결과 그래프 그리기
    plot_history(history, DATA_DIR)

In [ ]:
import os
import time
import copy 
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np # 🚨 CutMix를 위한 수학 라이브러리

# 🚨 [성능 폭발 레시피 1] RTX 5070의 텐서 코어를 100% 활용하기 위한 AMP 라이브러리
from torch.cuda.amp import autocast, GradScaler 

# ==========================================
# [PART 1] CMT 모델 아키텍처 직접 구현
# ==========================================
class LPU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
    def forward(self, x): return x + self.dwconv(x)

class LMHSA(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.sr_ratio = sr_ratio
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio, groups=dim)
            self.norm = nn.LayerNorm(dim)
        else:
            self.sr, self.norm = None, None

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        if self.sr is not None:
            x_2d = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_sr = self.norm(self.sr(x_2d).reshape(B, C, -1).permute(0, 2, 1))
            kv = self.kv(x_sr).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        else:
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        x = (attn.softmax(dim=-1) @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class IRFFN(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.conv1 = nn.Conv2d(in_features, hidden_features, 1, bias=False)
        self.bn1, self.act1 = nn.BatchNorm2d(hidden_features), nn.GELU()
        self.dwconv = nn.Conv2d(hidden_features, hidden_features, 3, 1, 1, groups=hidden_features, bias=False)
        self.bn2, self.act2 = nn.BatchNorm2d(hidden_features), nn.GELU()
        self.conv2 = nn.Conv2d(hidden_features, in_features, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(in_features)

    def forward(self, x):
        return x + self.bn3(self.conv2(self.act2(self.bn2(self.dwconv(self.act1(self.bn1(self.conv1(x))))))))

class CMTBlock(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio, expansion_ratio=4):
        super().__init__()
        self.lpu = LPU(dim)
        self.ln1 = nn.LayerNorm(dim)
        self.lmhsa = LMHSA(dim, num_heads, sr_ratio)
        self.ln2 = nn.LayerNorm(dim)
        self.irffn = IRFFN(dim, int(dim * expansion_ratio))

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.lpu(x)
        attn_out = self.lmhsa(self.ln1(x.flatten(2).transpose(1, 2)), H, W)
        x = x + attn_out.transpose(1, 2).reshape(B, C, H, W)
        return self.irffn(self.ln2(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W))

class PatchEmbed(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, 2, 2)
        self.norm = nn.LayerNorm(out_channels)
    def forward(self, x):
        x = self.proj(x)
        return self.norm(x.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(x.shape[0], x.shape[1], x.shape[2], x.shape[3])

class CMT_S(nn.Module):
    def __init__(self, num_classes=22):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.GELU()
        )
        self.patch_embed1, self.stage1 = PatchEmbed(32, 64), nn.Sequential(*[CMTBlock(64, 1, 8) for _ in range(3)])
        self.patch_embed2, self.stage2 = PatchEmbed(64, 128), nn.Sequential(*[CMTBlock(128, 2, 4) for _ in range(3)])
        self.patch_embed3, self.stage3 = PatchEmbed(128, 256), nn.Sequential(*[CMTBlock(256, 4, 2) for _ in range(16)])
        self.patch_embed4, self.stage4 = PatchEmbed(256, 512), nn.Sequential(*[CMTBlock(512, 8, 1) for _ in range(3)])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Conv2d(512, 1280, 1)
        self.head = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(self.patch_embed1(x))
        x = self.stage2(self.patch_embed2(x))
        x = self.stage3(self.patch_embed3(x))
        x = self.stage4(self.patch_embed4(x))
        x = self.head(self.proj(self.avgpool(x)).view(x.size(0), -1))
        return x

# ==========================================
# 💡 [핵심 추가 1] 프랑켄슈타인 데이터 합성 (CutMix) 
# ==========================================
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)

    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# ==========================================
# 💡 [핵심 추가 2] 평평한 골짜기 탐색기 (SAM Optimizer)
# ==========================================
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        
    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                p.add_(p.grad * scale.to(p))
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def restore_weights(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        return torch.norm(torch.stack([
            p.grad.norm(p=2).to(shared_device)
            for group in self.param_groups for p in group["params"] if p.grad is not None
        ]), p=2)

# ==========================================
# 💡 [핵심 추가 3] EMA (지수 이동 평균) 클래스 
# ==========================================
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        for param in self.ema.parameters():
            param.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_param, param in zip(self.ema.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1 - self.decay)

# ==========================================
# [PART 2] 훈련 함수 (CutMix + SAM + EMA + 마라톤 모드 결합)
# ==========================================
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, epochs, save_path):
    best_acc = 0.0
    ema = ModelEMA(model, decay=0.999)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    print(f"\n🔥 [성능 부스트 끝판왕] CutMix + SAM + EMA 장착 완료!")
    print(f"🔥 조기 종료 없이 {epochs} Epoch 마라톤을 완주합니다!\n")
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # --- 훈련 (Train) ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Train]")
        
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # 💡 CutMix (50% 확률로 데이터 합성)
            r = np.random.rand(1)
            apply_cutmix = r < 0.5
            
            if apply_cutmix:
                lam = np.random.beta(1.0, 1.0)
                rand_index = torch.randperm(inputs.size()[0]).to(device)
                target_a = labels
                target_b = labels[rand_index]
                bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
                inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
                lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size()[-1] * inputs.size()[-2]))
            
            # 💡 SAM 1단계 (구덩이 탐색) + AMP 적용
            with autocast():
                outputs = model(inputs)
                if apply_cutmix:
                    loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
                else:
                    loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer.base_optimizer) # 그래디언트 스케일 해제
            optimizer.first_step(zero_grad=True)      # 불안정한 곳으로 1보 전진
            
            # 💡 SAM 2단계 (진짜 업데이트) + AMP 적용
            with autocast():
                outputs = model(inputs)
                if apply_cutmix:
                    loss2 = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
                else:
                    loss2 = criterion(outputs, labels)
            
            scaler.scale(loss2).backward()
            optimizer.restore_weights()               # 전진했던 것 복구
            scaler.step(optimizer.base_optimizer)     # 안전한 방향으로 진짜 업데이트
            scaler.update()
            
            # EMA 두뇌 업데이트
            ema.update(model)
            
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            
            # 정확도 계산 (CutMix 시에는 비율이 더 높은 쪽을 정답으로 간주)
            if apply_cutmix:
                true_labels = target_a if lam > 0.5 else target_b
                train_correct += predicted.eq(true_labels).sum().item()
            else:
                train_correct += predicted.eq(labels).sum().item()
                
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{train_correct/train_total:.4f}"})
            
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        
        # --- 검증 (Validation) ---
        ema.ema.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Val]  ")
        
        with torch.no_grad():
            for inputs, labels in val_pbar:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # TTA (Test-Time Augmentation)
                inputs_flipped = torch.flip(inputs, dims=[3])
                
                with autocast():
                    outputs_orig = ema.ema(inputs)
                    outputs_flip = ema.ema(inputs_flipped)
                    outputs = (outputs_orig + outputs_flip) / 2.0 
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                val_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{val_correct/val_total:.4f}"})
                
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        scheduler.step()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        elapsed_time = time.time() - start_time
        current_lr = optimizer.base_optimizer.param_groups[0]['lr'] # SAM 내부 optimizer의 LR 참조
        print(f"✅ Epoch {epoch+1:03d} 완료 | Time: {elapsed_time:.0f}s | LR: {current_lr:.6f} | "
              f"Train Acc: {epoch_train_acc:.4f} | EMA Val Acc: {epoch_val_acc:.4f}")
        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(ema.ema.state_dict(), save_path)
            print(f"  🌟 최고 성능 갱신! 황금비율 EMA 모델 저장됨: {save_path}\n")
        else:
            print(f"  ⚠️ 성능 갱신 실패 (하지만 조기 종료 없이 {epochs} Epoch까지 꿋꿋하게 완주합니다!)\n")

    print(f"\n🎉 대장정 종료! 최고 검증 정확도 (EMA): {best_acc:.4f}")
    return history

def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('CMT Model Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('CMT Model Loss')
    plt.legend()
    graph_path = os.path.join(save_dir, "training_log.png")
    plt.savefig(graph_path)
    print(f"📊 학습 그래프가 저장되었습니다: {graph_path}")
    plt.show()

# ==========================================
# [PART 3] 메인 실행부
# ==========================================
if __name__ == "__main__":
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
    DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE

    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_cmt_model.pt")

    BATCH_SIZE = 64 
    EPOCHS = 100    
    LEARNING_RATE = 5e-4 
    NUM_WORKERS = 0 

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 학습 장치: {device} 로 구동됩니다.")

    common_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("📂 데이터셋을 불러오는 중입니다...")
    train_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'train'), transform=common_transform)
    val_dataset = datasets.ImageFolder(root=os.path.join(DATA_DIR, 'val'), transform=common_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    NUM_CLASSES = len(train_dataset.classes)
    
    print("🧠 자체 구현한 오리지널 CMT-S 모델을 로딩합니다...")
    model = CMT_S(num_classes=NUM_CLASSES).to(device)

    # 💡 라벨 스무딩 적용
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # 💡 기본 AdamW를 SAM 옵티마이저로 감싸기
    base_optimizer = optim.AdamW
    optimizer = SAM(model.parameters(), base_optimizer, lr=LEARNING_RATE, weight_decay=5e-4) 
    
    # 💡 스케줄러 (SAM의 base_optimizer를 바라보도록 설정)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer.base_optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # AMP 스케일러 생성
    scaler = GradScaler()

    # 훈련 함수 호출
    history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, EPOCHS, MODEL_SAVE_PATH)
    
    plot_history(history, DATA_DIR)